# Leer los datos de la carpeta y representarlos.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ruta_actual = Path(__file__).resolve().parent

# 2. Sube dos carpetas hacia arriba 
ruta_archivo = ruta_actual.parent.parent / "17h55m39s-12-May.txt"
# Leer el archivo sin encabezados
df = pd.read_csv(
    ruta_archivo,
    sep=r'\s+',  # Separador
    header=None,  # Sin encabezados
    names=['tiempo', 'col2', 'actividad_extracelular', 'intra_LP', 'intra_PD', 'col6'],
    usecols=['tiempo', 'actividad_extracelular', 'intra_LP', 'intra_PD'] 
)

print(df.head())
print(f"\nDimensiones: {df.shape}")

In [ ]:
# Visualización

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(df['tiempo']-df['tiempo'].min(), df['actividad_extracelular'], label='Actividad Extracelular')
plt.plot(df['tiempo']-df['tiempo'].min(), df['intra_LP'], label='Neurona LP')
plt.plot(df['tiempo']-df['tiempo'].min(), df['intra_PD'], label='Neurona PD')
plt.xlabel('Tiempo (ms)')
plt.ylabel('Voltaje (mV)')
plt.title('Datos de Actividad Neuronal')
plt.legend()
plt.show()

In [ ]:
puntos_grafica = 500000

In [ ]:
# 500000 puntos es un número grande.

plt.figure(figsize=(12, 6))
plt.plot(df['tiempo'][:puntos_grafica]-df['tiempo'][:puntos_grafica].min(), df['actividad_extracelular'][:puntos_grafica], label='Actividad Extracelular')
plt.plot(df['tiempo'][:puntos_grafica]-df['tiempo'][:puntos_grafica].min(), df['intra_LP'][:puntos_grafica], label='Neurona LP')
plt.plot(df['tiempo'][:puntos_grafica]-df['tiempo'][:puntos_grafica].min(), df['intra_PD'][:puntos_grafica], label='Neurona PD')
plt.xlabel('Tiempo (ms)')        
plt.ylabel('Voltaje (mV)')
plt.title('Datos de Actividad Neuronal (Primeros 500000 puntos)')
plt.legend()
plt.show()

In [ ]:
puntos_grafica = 15000

In [ ]:
# Voy a hacer la representación de nuevo pero con solo un fragmento porque ahora no se entiende nada.

plt.figure(figsize=(12, 6))
plt.plot(df['tiempo'][:puntos_grafica], df['actividad_extracelular'][:puntos_grafica], label='Actividad Extracelular')
plt.plot(df['tiempo'][:puntos_grafica], df['intra_LP'][:puntos_grafica], label='Intra LP')
plt.plot(df['tiempo'][:puntos_grafica], df['intra_PD'][:puntos_grafica], label='Intra PD')
plt.xlabel('Tiempo (ms)')        
plt.ylabel('Voltaje (mV)')
plt.title('Datos de Actividad Neuronal (Primeros 15000 puntos)')
plt.legend()
plt.show()

# Aplicar filtros de media 


In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt
from scipy import signal as sp_signal

def filtro_media_movil_pandas(data, window_size):
    """
    Aplica media móvil manteniendo la longitud original.
    Los bordes usan menos puntos pero no hay artefactos severos.
    """
    import pandas as pd
    
    if not isinstance(data, pd.Series):
        data = pd.Series(data)
    
    # center=True centra la ventana
    # min_periods=1 permite calcular incluso con pocos puntos
    y = data.rolling(window=window_size, center=True, min_periods=1).mean()
    
    return y.values




def eliminar_tendencia(señal, metodo='linear', grado=2):
    """
    Elimina la tendencia de la señal manteniendo la amplitud original.
    
    Parámetros:
    -----------
    señal : array
        Señal original
    metodo : str
        'linear' - elimina tendencia lineal (recomendado para deriva constante)
        'polynomial' - elimina tendencia polinómica (para deriva no lineal)
    grado : int
        Grado del polinomio (solo para metodo='polynomial')
    """
    if metodo == 'linear':
        # Eliminar tendencia lineal - mantiene amplitud
        señal_detrend = sp_signal.detrend(señal, type='linear')
        
    
    return señal_detrend


In [ ]:
lista_puntos = [1,20,30,35,40,50,55,60,65,70,75,80,100,150,200]
puntos_grafica = 20000

for window in lista_puntos:
    señal_filt = filtro_media_movil_pandas(df['intra_LP'], window)
    
    plt.figure(figsize=(12, 4))
    if window == 1:
        plt.plot(df['tiempo'][:puntos_grafica], df['intra_LP'][:puntos_grafica], 
                 label='Intra LP Original', alpha=0.8, linewidth=0.5)
        #plt.plot(df['tiempo'][:puntos_grafica], df['intra_LP_filtrada'][:puntos_grafica], 
        #     label='LP Butterworth', linewidth=2)
        
    else:
        #plt.plot(df['tiempo'][:puntos_grafica], df['intra_LP'][:puntos_grafica], 
        #         label='Intra LP Original', alpha=0.8, linewidth=0.5)
        #plt.plot(df['tiempo'][:puntos_grafica], df['intra_LP_filtrada'][:puntos_grafica], 
        #         label='LP Butterworth', linewidth=2)
        plt.plot(df['tiempo'][:puntos_grafica], señal_filt[:puntos_grafica], 
                 label=f'Intra LP Media Móvil (ventana={window})', linewidth=1)
    plt.xlabel('Tiempo (s)')
    plt.ylabel('Señal')
    plt.title(f'Filtro Media Móvil - Ventana {window}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
lista_puntos = lista_puntos = [1,20,30,35,40,50,55,60,65,70,75,80,100,150,200]

for window in lista_puntos:
    señal_filt = filtro_media_movil_pandas(df['intra_PD'], window)
    
    plt.figure(figsize=(12, 4))
    plt.plot(df['tiempo'][:puntos_grafica], df['intra_PD'][:puntos_grafica], 
             label='Intra PD Original', alpha=0.1, linewidth=0.5)
    plt.plot(df['tiempo'][:puntos_grafica], señal_filt[:puntos_grafica], 
             label=f'Intra PD Media Móvil (ventana={window})', linewidth=1)
    plt.xlabel('Tiempo (s)')
    plt.ylabel('Señal')
    plt.title(f'Filtro Media Móvil - Ventana {window}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## Selecciono únicamente la mejor ventana:

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt
from scipy import signal as sp_signal
def detrend_mantener_nivel(señal):
    """
    Elimina tendencia pero mantiene el nivel medio original
    """
    media_original = np.mean(señal)
    señal_detrend = sp_signal.detrend(señal, type='linear')
    señal_ajustada = señal_detrend + media_original
    return señal_ajustada

def filtro_media_movil_pandas(data, window_size):
    """
    Aplica media móvil manteniendo la longitud original.
    Los bordes usan menos puntos pero no hay artefactos severos.
    """
    import pandas as pd
    
    if not isinstance(data, pd.Series):
        data = pd.Series(data)
    
    # center=True centra la ventana
    # min_periods=1 permite calcular incluso con pocos puntos
    y = data.rolling(window=window_size, center=True, min_periods=1).mean()
    
    return y.values



In [ ]:
ventana1 = 30
ventana2 = 30
fc = 0.5

#Filtrramos tambien con el filtro paso alto


señal_filt_LP = filtro_media_movil_pandas(df['intra_LP'], ventana1)
señal_filt_PD = filtro_media_movil_pandas(df['intra_PD'], ventana2)


señal_filt_LP = detrend_mantener_nivel(señal_filt_LP)
señal_filt_PD = detrend_mantener_nivel(señal_filt_PD)

#señal_filt_LP = detrend_mantener_nivel(df['intra_LP'])
#señal_filt_PD = detrend_mantener_nivel(df['intra_PD'])


df['intra_LP_media_movil'] = señal_filt_LP
df['intra_PD_media_movil'] = señal_filt_PD


plt.figure(figsize=(12, 6))

plt.plot(df['tiempo'][550000:600000], señal_filt_PD[550000:600000], label='Intra PD Media Móvil')
plt.xlabel('Tiempo (s)')
plt.ylabel('Voltaje (mV)')
plt.title('Datos de Actividad Neuronal - Filtro Media Móvil')



plt.figure(figsize=(12, 6))
plt.plot(df['tiempo'], señal_filt_LP, label='Intra LP Media Móvil')
plt.plot(df['tiempo'], señal_filt_PD, label='Intra PD Media Móvil')
plt.xlabel('Tiempo (s)')
plt.ylabel('Voltaje (mV)')
plt.title('Datos de Actividad Neuronal- Después de eliminar tendencia y aplicar filtro de media móvil')

In [ ]:
# Voy a hacer la representación de nuevo pero con solo un fragmento porque ahora no se entiende nada.

puntos_grafica = 15000

plt.figure(figsize=(12, 6))

plt.plot(df['tiempo'][:puntos_grafica], señal_filt_LP[:puntos_grafica], label='Neurona LP', alpha=0.9)

plt.plot(df['tiempo'][:puntos_grafica], señal_filt_PD[:puntos_grafica]+2, label='Neurona PD', alpha=0.9)

plt.xlabel('Tiempo (ms)')        

plt.ylabel('Voltaje (mV)')

plt.title('Datos de Actividad Neuronal (Primeros 10000 puntos)')


plt.grid(False)

plt.show()

# Detección de eventos

In [ ]:
from scipy.signal import find_peaks
import numpy as np
señal_LP = df['intra_LP_media_movil'].values
señal_PD = df['intra_PD_media_movil'].values


# Umbrales
umbral_inf_LP = -5.76
umbral_max_LP = -4.2



umbral_inf_PD = -8.3
umbral_max_PD = -7.1
umbral_maxU_PD = -6.6



dt = 0.1

peaks_LP, props_LP = find_peaks(señal_LP, height=umbral_max_LP, distance=150, )
hiper_indices_LP, props_hiper_LP = find_peaks(-señal_LP, height=-umbral_inf_LP, distance=3000,width=100)


peaks_PD, props_PD = find_peaks(señal_PD, height=umbral_max_PD, distance=150, )
hiper_indices_PD, props_hiper_PD = find_peaks(-señal_PD, height=-umbral_inf_PD, distance=5000,width=200)


In [ ]:
import numpy as np

def detectar_rafagas(peaks, umbral_intra, signal, umbral_amplitud=None, min_picos=2):
    """
    Detecta el primer y último pico de cada ráfaga.
    
    Parámetros:
    -----------
    peaks            : array de índices de picos (ordenados)
    umbral_intra     : distancia máxima entre picos para considerarlos
                       de la MISMA ráfaga. Si el gap es mayor → nueva ráfaga.
    signal           : señal original (para consultar amplitudes)
    umbral_amplitud  : amplitud mínima que debe superar el pico final.
                       Si None, se usa el último pico del grupo (comportamiento original).
    min_picos        : número mínimo de picos para que sea una ráfaga válida
                       (descarta picos aislados o grupos muy pequeños)
    
    Retorna:
    --------
    rafagas: lista de dicts con 'inicio', 'fin', 'n_picos', 'picos'
    """
    
    gaps = np.diff(peaks)
    cortes = np.where(gaps > umbral_intra)[0]
    
    # Construir grupos
    grupos = []
    inicio_idx = 0
    for corte in cortes:
        grupos.append(peaks[inicio_idx : corte + 1])
        inicio_idx = corte + 1
    grupos.append(peaks[inicio_idx:])
    
    # Filtrar y construir ráfagas
    rafagas = []
    for grupo in grupos:
        if len(grupo) < min_picos:
            continue
        
        # Determinar el pico final
        if umbral_amplitud is not None:
            # Buscar el ÚLTIMO pico del grupo que supere el umbral de amplitud
            amplitudes = signal[grupo]
            indices_validos = np.where(amplitudes >= umbral_amplitud)[0]
            
            if len(indices_validos) == 0:
                continue
            
            pico_fin = grupo[indices_validos[-1]]   # último que supera el umbral
        else:
            pico_fin = grupo[-1]  # comportamiento original
        
        rafagas.append({
            'inicio':  grupo[0],
            'fin':     pico_fin,
            'n_picos': len(grupo),
            'picos':   grupo
        })
    
    return rafagas


In [ ]:


UMBRAL_INTRA = 800   # gap máximo entre picos de la misma ráfaga (en muestras)
MIN_PICOS    = 3     # mínimo de picos para considerar ráfaga válida

rafagas = detectar_rafagas(peaks_LP, signal = señal_LP,umbral_intra=UMBRAL_INTRA, min_picos=MIN_PICOS)

# Ver resultados
for i, r in enumerate(rafagas):
    print(f"Ráfaga {i+1:3d} | inicio: {r['inicio']:10d} | fin: {r['fin']:10d} | picos: {r['n_picos']}")

# Extraer solo los potenciales de inicio y fin de la ráfaga
fSp_indices_LP = np.array([r['inicio'] for r in rafagas])
lSp_indices_LP   = np.array([r['fin']   for r in rafagas])



# Ajusta estos dos valores según tu señal:
UMBRAL_INTRA_PD = 400   # gap máximo entre picos de la misma ráfaga (en muestras)
rafagas = detectar_rafagas(peaks_PD, signal=señal_PD, umbral_intra=UMBRAL_INTRA_PD, min_picos=MIN_PICOS, umbral_amplitud=umbral_maxU_PD)

# Ver resultados
for i, r in enumerate(rafagas):
    print(f"Ráfaga {i+1:3d} | inicio: {r['inicio']:10d} | fin: {r['fin']:10d} | picos: {r['n_picos']}")

# Extraer solo los potenciales de inicio y fin de la ráfaga
fSp_indices_PD = np.array([r['inicio'] for r in rafagas])
lSp_indices_PD   = np.array([r['fin']   for r in rafagas])



In [ ]:
def plot_signal_subplots(t_irr, x_irr, min_t, min_x, max_t, max_x, maxU_t, maxU_x,
                         window_size=5, step=5, cols=2):
    """
    Genera subplots con ventanas temporales de la señal.
    """
    # Asegurar que son arrays de NumPy
    min_t = np.asarray(min_t)
    min_x = np.asarray(min_x)
    max_t = np.asarray(max_t)
    max_x = np.asarray(max_x)
    maxU_t = np.asarray(maxU_t)
    maxU_x = np.asarray(maxU_x)
    
    # DEBUG: Verificar dimensiones
    print(f"Dimensiones:")
    print(f"  min_t: {min_t.shape}, min_x: {min_x.shape}")
    print(f"  max_t: {max_t.shape}, max_x: {max_x.shape}")
    print(f"  maxU_t: {maxU_t.shape}, maxU_x: {maxU_x.shape}")
    
    # Verificar que las dimensiones coincidan
    assert len(min_t) == len(min_x), f"min_t y min_x tienen diferentes tamaños: {len(min_t)} vs {len(min_x)}"
    assert len(max_t) == len(max_x), f"max_t y max_x tienen diferentes tamaños: {len(max_t)} vs {len(max_x)}"
    assert len(maxU_t) == len(maxU_x), f"maxU_t y maxU_x tienen diferentes tamaños: {len(maxU_t)} vs {len(maxU_x)}"
    
    t_total = t_irr[-1]
    n_windows = int(np.ceil(t_total / step))
    
    # Calcular filas necesarias
    rows = int(np.ceil(n_windows / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(12*cols, 4*rows))
    axes = axes.flatten() if n_windows > 1 else [axes]
    
    for i in range(n_windows):
        t_start = i * step
        t_end = min(t_start + window_size, t_total)
        
        if t_start >= t_total:
            break
        
        ax = axes[i]
        
        # Filtrar datos de la señal
        mask_signal = (t_irr >= t_start) & (t_irr <= t_end)
        
        # Para los puntos detectados, filtrar solo aquellos en la ventana
        indices_min = np.where((min_t >= t_start) & (min_t <= t_end))[0]
        indices_max = np.where((max_t >= t_start) & (max_t <= t_end))[0]
        indices_maxU = np.where((maxU_t >= t_start) & (maxU_t <= t_end))[0]
        
        # Graficar señal
        ax.plot(t_irr[mask_signal], x_irr[mask_signal], alpha=0.7, label='x(t)')
        
        # Graficar puntos detectados
        if len(indices_min) > 0:
            ax.plot(min_t[indices_min], min_x[indices_min], 'o', color='red', 
                   markersize=4, label='Mínimos')
        
        if len(indices_max) > 0:
            # DEBUG en caso de error
            if indices_max.max() >= len(max_x):
                print(f"ERROR en ventana {i}: max índice={indices_max.max()}, len(max_x)={len(max_x)}")
                print(f"indices_max problemáticos: {indices_max[indices_max >= len(max_x)]}")
                # Filtrar índices válidos
                indices_max = indices_max[indices_max < len(max_x)]
            
            if len(indices_max) > 0:
                ax.plot(max_t[indices_max], max_x[indices_max], 'o', color='green', 
                       markersize=4, label='Máximos')
        
        if len(indices_maxU) > 0:
            # DEBUG en caso de error
            if indices_maxU.max() >= len(maxU_x):
                print(f"ERROR en ventana {i}: maxU índice={indices_maxU.max()}, len(maxU_x)={len(maxU_x)}")
                indices_maxU = indices_maxU[indices_maxU < len(maxU_x)]
            
            if len(indices_maxU) > 0:
                ax.plot(maxU_t[indices_maxU], maxU_x[indices_maxU], 'o', color='orange', 
                       markersize=4, label='Máximos previos')
        
        ax.set_xlabel("Tiempo (t)")
        ax.set_ylabel("x(t)")
        ax.set_title(f"t ∈ [{t_start:.1f}, {t_end:.1f}]")
        ax.set_xlim(t_start, t_end)
        ax.grid(True)
        
        # Añadir leyenda solo en el primer subplot
        if i == 0:
            ax.legend(fontsize=8)
    
    # Ocultar subplots vacíos
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()



## Validar eventos en un fragmento

In [ ]:

puntos_grafica_final = 1550000
puntos_inicio = 1500000

# Filtrar los índices.
fsp_mask = (fSp_indices_LP < puntos_grafica_final) & (fSp_indices_LP >= puntos_inicio)
lsp_mask = (lSp_indices_LP < puntos_grafica_final) & (lSp_indices_LP >= puntos_inicio)
hiper_mask = (hiper_indices_LP < puntos_grafica_final) & (hiper_indices_LP >= puntos_inicio)

# Índices relativos al fragmento
fsp_rel = fSp_indices_LP[fsp_mask] - puntos_inicio
lsp_rel = lSp_indices_LP[lsp_mask] - puntos_inicio
hiper_rel = hiper_indices_LP[hiper_mask] - puntos_inicio

plt.figure(figsize=(15, 6))

# Datos del fragmento
tiempo_frag = df['tiempo'].values[puntos_inicio:puntos_grafica_final]
t_eje = tiempo_frag - tiempo_frag.min() # Tiempo relativo para el eje X (empieza en 0)
señal_frag_filt = señal_filt_LP[puntos_inicio:puntos_grafica_final]
señal_frag_orig = df['intra_LP'].values[puntos_inicio:puntos_grafica_final]

ymin, ymax = señal_frag_orig.min(), señal_frag_orig.max()

# Graficar señales
plt.plot(t_eje, señal_frag_filt, label='Neurona LP (filtrada)', color='tab:blue', alpha=1, linewidth=1.5)
plt.plot(t_eje, señal_frag_orig, label='Neurona LP (original)', color='tab:purple', alpha=0.9)


if len(fsp_rel) > 0:
    plt.vlines(t_eje[fsp_rel], ymin, ymax, colors='red', linestyles='--', 
               alpha=0.8, label='Primer potencial')

if len(lsp_rel) > 0:
    plt.vlines(t_eje[lsp_rel], ymin, ymax, colors='orange', linestyles='--', 
               alpha=0.8, label='Último potencial')

if len(hiper_rel) > 0:
    plt.vlines(t_eje[hiper_rel], ymin, ymax, colors='green', linestyles='--', 
               alpha=0.8, label='Hiperpolarización')

plt.xlabel('Tiempo (ms)')        
plt.ylabel('Voltaje (mV)')
plt.title(f'Validación de eventos en la señal filtrada de la neurona LP (fragmento de 5 segundos)')
plt.legend(loc='upper right', frameon=True)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### Representación entera total de los eventos para validar su correcta detección

In [ ]:

t_irr = np.arange(len(señal_LP)) * dt
hiper_t_LP = t_irr[hiper_indices_LP]
Amp_hiper_LP = señal_LP[hiper_indices_LP]

fSp_t_LP = t_irr[fSp_indices_LP]
Amp_fSp_LP = señal_LP[fSp_indices_LP]

lSp_t_LP = t_irr[lSp_indices_LP]
Amp_lSp_LP = señal_LP[lSp_indices_LP]

t_irr_p = np.arange(len(df["intra_LP"])) * dt
# --- Uso ---
plot_signal_subplots(t_irr,señal_LP, hiper_t_LP, Amp_hiper_LP, fSp_t_LP, Amp_fSp_LP, lSp_t_LP, Amp_lSp_LP,
                    window_size=5000, step=5000, cols=3) 

In [ ]:

puntos_grafica_final = 1550000
puntos_inicio = 1500000

fsp_mask = (fSp_indices_PD < puntos_grafica_final) & (fSp_indices_PD >= puntos_inicio)
lsp_mask = (lSp_indices_PD < puntos_grafica_final) & (lSp_indices_PD >= puntos_inicio)
hiper_mask = (hiper_indices_PD < puntos_grafica_final) & (hiper_indices_PD >= puntos_inicio)


fsp_rel = fSp_indices_PD[fsp_mask] - puntos_inicio
lsp_rel = lSp_indices_PD[lsp_mask] - puntos_inicio
hiper_rel = hiper_indices_PD[hiper_mask] - puntos_inicio

plt.figure(figsize=(15, 6))

# Datos del fragmento
tiempo_frag = df['tiempo'].values[puntos_inicio:puntos_grafica_final]
t_eje = tiempo_frag - tiempo_frag.min() # Tiempo relativo para el eje X (empieza en 0)
señal_frag_filt = señal_filt_PD[puntos_inicio:puntos_grafica_final]
señal_frag_orig = df['intra_PD'].values[puntos_inicio:puntos_grafica_final]

ymin, ymax = señal_frag_orig.min(), señal_frag_orig.max()

# Graficar señales
plt.plot(t_eje, señal_frag_filt, label='Neurona PD (filtrada)', color='tab:blue', alpha=1, linewidth=1.5)
plt.plot(t_eje, señal_frag_orig, label='Neurona PD (original)', color='tab:purple', alpha=0.9)

if len(fsp_rel) > 0:
    plt.vlines(t_eje[fsp_rel], ymin, ymax, colors='red', linestyles='--', 
               alpha=0.8, label='Primer potencial')

if len(lsp_rel) > 0:
    plt.vlines(t_eje[lsp_rel], ymin, ymax, colors='orange', linestyles='--', 
               alpha=0.8, label='Último potencial')

if len(hiper_rel) > 0:
    plt.vlines(t_eje[hiper_rel], ymin, ymax, colors='green', linestyles='--', 
               alpha=0.8, label='Hiperpolarización')

plt.xlabel('Tiempo (ms)')        
plt.ylabel('Voltaje (mV)')
plt.title(f'Validación de eventos en la señal filtrada de la neurona PD (fragmento de 5 segundos)')
plt.legend(loc='upper right', frameon=True)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Hay un outlier un caso raro error bug de la señal

In [ ]:

t_irr = np.arange(len(señal_LP)) * dt
hiper_t_PD = t_irr[hiper_indices_PD]
Amp_hiper_PD = señal_PD[hiper_indices_PD]

fSp_t_PD = t_irr[fSp_indices_PD]
Amp_fSp_PD = señal_PD[fSp_indices_PD]

lSp_t_PD = t_irr[lSp_indices_PD]
Amp_lSp_PD = señal_PD[lSp_indices_PD]

t_irr_p = np.arange(len(df["intra_LP"])) * dt


plot_signal_subplots(t_irr,señal_PD, hiper_t_PD, Amp_hiper_PD, fSp_t_PD, Amp_fSp_PD, lSp_t_PD, Amp_lSp_PD,
                    window_size=5000, step=5000, cols=3)    

In [ ]:
# Plot ocmbinado.
import numpy as np
import matplotlib.pyplot as plt

def plot_combined_signals_subplots(t_irr, x_lp, min_t_lp, min_x_lp, max_t_lp, max_x_lp, maxU_t_lp, maxU_x_lp,
                                   x_pd, min_t_pd, min_x_pd, max_t_pd, max_x_pd, maxU_t_pd, maxU_x_pd,
                                   window_size=5000, step=5000, cols=2):
    """
    Genera subplots con ventanas temporales donde se superponen las señales LP y PD.
    """
    t_total = t_irr[-1]
    n_windows = int(np.ceil(t_total / step))
    rows = int(np.ceil(n_windows / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 2 * rows), sharey=False)
    axes = axes.flatten() if n_windows > 1 else [axes]
    
    for i in range(n_windows):
        t_start = i * step
        t_end = min(t_start + window_size, t_total)
        if t_start >= t_total: break
        
        ax = axes[i]
        mask = (t_irr >= t_start) & (t_irr <= t_end)
        
        # --- GRAFICAR SEÑAL LP ---
        ax.plot(t_irr[mask], x_lp[mask], color='blue', alpha=0.5, label='LP' if i==0 else "")
        # Puntos LP (Mínimos, Máximos, Máximos previos)
        idx_min_lp = np.where((min_t_lp >= t_start) & (min_t_lp <= t_end))[0]
        ax.scatter(min_t_lp[idx_min_lp], min_x_lp[idx_min_lp], color='darkblue', s=10, zorder=3)
        
        # --- GRAFICAR SEÑAL PD ---
        ax.plot(t_irr[mask], x_pd[mask], color='orange', alpha=0.5, label='PD' if i==0 else "")
        # Puntos PD
        idx_min_pd = np.where((min_t_pd >= t_start) & (min_t_pd <= t_end))[0]
        ax.scatter(min_t_pd[idx_min_pd], min_x_pd[idx_min_pd], color='darkorange', s=10, zorder=3)

        # Configuración de los ejes
        ax.set_title(f"Ventana {t_start} - {t_end}")
        ax.set_xlim(t_start, t_end)
        ax.grid(True, linestyle='--', alpha=0.6)
        if i == 0: ax.legend(loc='upper right')

    # Ocultar sobrantes
    for j in range(i + 1, len(axes)): axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

# Llamada a la función con ambas señales
plot_combined_signals_subplots(
    t_irr, 
    señal_LP, hiper_t_LP, Amp_hiper_LP, fSp_t_LP, Amp_fSp_LP, lSp_t_LP, Amp_lSp_LP, # Datos LP
    señal_PD, hiper_t_PD, Amp_hiper_PD, fSp_t_PD, Amp_fSp_PD, lSp_t_PD, Amp_lSp_PD, # Datos PD
    window_size=5000, step=5000, cols=1 # Cols=1 suele ser mejor para ver la relación temporal
)

#

## Construcción de intervalos bsados ´únicamente en potenciales de acción .

In [ ]:
def periodo(tiempos_picos, dt):
    """
    Calcula periodos entre picos consecutivos.
    Robusto: maneja correctamente la secuencia temporal.
    """
    if len(tiempos_picos) < 2:
        return [], []
    
    periodos = []
    timestamps = []
    
    
    
    for i in range(1, len(tiempos_picos)):
        periodo = (tiempos_picos[i] - tiempos_picos[i-1]) * dt
        if periodo > 0:  # Validación adicional
            periodos.append(periodo)
            timestamps.append(tiempos_picos[i] * dt)  # Tiempo del pico actual
    
    return periodos, timestamps


def burstDur(tiempos_primer_pico, tiempos_ultimo_pico, dt):
    """
    Calcula duración de ráfagas (primer pico a último pico).
    Robusto: asegura correspondencia temporal correcta entre primeros y últimos picos.
    """
    if len(tiempos_primer_pico) == 0 or len(tiempos_ultimo_pico) == 0:
        return [], []
    
    duraciones = []
    timestamps = []
    
    # Índice para recorrer últimos picos
    j = 0
    
    for i, t_inicio in enumerate(tiempos_primer_pico):
        # Buscar el próximo último pico que viene DESPUÉS del primer pico
        while j < len(tiempos_ultimo_pico) and tiempos_ultimo_pico[j] <= t_inicio:
            j += 1
        
        if j < len(tiempos_ultimo_pico):
            t_fin = tiempos_ultimo_pico[j]
            duracion = (t_fin - t_inicio) * dt
            
            if duracion > 0:  # Validación: duración debe ser positiva
                duraciones.append(duracion)
                timestamps.append(t_inicio * dt)  # Tiempo de inicio de la ráfaga
            j += 1  # Avanzar al siguiente último pico para la siguiente ráfaga
        else:
            break  # No hay más últimos picos disponibles
    
    return duraciones, timestamps


def delay(tiempos_fin_n1, tiempos_inicio_n2, dt):
    """
    Calcula delays desde último pico de N1 hasta primer pico de N2.
    Robusto: asegura que el pico de N2 viene DESPUÉS del pico de N1.
    
    Parámetros:
    -----------
    tiempos_fin_n1 : lista de tiempos
        Últimos picos de la neurona 1
    tiempos_inicio_n2 : lista de tiempos
        Primeros picos de la neurona 2
    """
    if len(tiempos_fin_n1) == 0 or len(tiempos_inicio_n2) == 0:
        return [], []
    
    delays = []
    timestamps = []
    
    # Índice para recorrer primeros picos de N2
    j = 0
    
    for t_fin_n1 in tiempos_fin_n1:
        # Buscar el próximo primer pico de N2 que viene DESPUÉS del último pico de N1
        while j < len(tiempos_inicio_n2) and tiempos_inicio_n2[j] <= t_fin_n1:
            j += 1
        
        if j < len(tiempos_inicio_n2):
            t_inicio_n2 = tiempos_inicio_n2[j]
            delay_t = (t_inicio_n2 - t_fin_n1) * dt
            
            if delay_t > 0:  # Validación: delay debe ser positivo
                delays.append(delay_t)
                timestamps.append(t_fin_n1 * dt)  # Tiempo del último pico de N1
            j += 1  # Avanzar al siguiente primer pico de N2
        else:
            break  # No hay más primeros picos de N2 disponibles
    
    return delays, timestamps

def Interval_N(burst_durations, delays, timestamps_burst):
    """Suma Burst Duration + Delay, usa timestamp del burst"""
    intervals = []
    timestamps = []
    for i in range(min(len(burst_durations), len(delays))):
        intervals.append(burst_durations[i] + delays[i])
        timestamps.append(timestamps_burst[i])  # Usa el timestamp del burst
    return intervals, timestamps


def Interval_xN(burst_durations_n1, delays_n2, timestamps_burst_n1):
    """Intervalo desde último pico n1 al último pico n2"""
    intervals = []
    timestamps = []
    for i in range(min(len(burst_durations_n1), len(delays_n2))):
        intervals.append(burst_durations_n1[i] + delays_n2[i])
        timestamps.append(timestamps_burst_n1[i])  # Usa timestamp del burst n1
    return intervals, timestamps



In [ ]:
LP_fst2fst,periodotimestamp = periodo(fSp_indices_LP, dt) #Se comparan con el fst2fst 
PD_fst2fst,periodotimestamp_PD = periodo(fSp_indices_PD, dt)

print(f"Periodo LP: {LP_fst2fst[:5]} (timestamps: {periodotimestamp[:5]})")
print(f"Periodo PD: {PD_fst2fst[:5]} (timestamps: {periodotimestamp_PD[:5]})")

In [ ]:
LP_spkperiod, spkperiodtimestamp = burstDur(fSp_indices_LP,lSp_indices_LP, dt) #Se comparan con el spkperiod
PD_spkperiod, spkperiodtimestamp_PD = burstDur(fSp_indices_PD,lSp_indices_PD, dt)
print(f"Spike Duration LP: {LP_spkperiod[:5]} (timestamps: {spkperiodtimestamp[:5]})")
print(f"Spike Duration PD: {PD_spkperiod[:5]} (timestamps: {spkperiodtimestamp_PD[:5]})")

In [ ]:

Plateau, LP_PD_Delay_Timestamps = delay(lSp_indices_LP, fSp_indices_PD, dt) #Plateau se compara con el delay entre el último pico de LP y el primer pico de PD
PD_LP_Delay, PD_LP_Delay_Timestamps = delay(lSp_indices_PD, fSp_indices_LP, dt)

print(f"Delay LP->PD: {Plateau[:5]} (timestamps: {LP_PD_Delay_Timestamps[:5]})")
print(f"Delay PD->LP: {PD_LP_Delay[:5]} (timestamps: {PD_LP_Delay_Timestamps[:5]})")

In [ ]:


LP_Interval_N, LP_Interval_N_Timestamps = Interval_N(LP_spkperiod, Plateau, timestamps_burst=spkperiodtimestamp) #Interval N se compara con la suma de spkperiod + delay entre el último pico de LP y el primer pico de PD


#Este es el LPPDlspkperiod
PD_Interval_N, PD_Interval_N_Timestamps = Interval_N(PD_spkperiod, PD_LP_Delay, timestamps_burst=spkperiodtimestamp_PD) #Interval N se compara con la suma de spkperiod + delay entre el último pico de PD y el primer pico de LP

print(f"Interval N LP: {LP_Interval_N[:5]} (timestamps: {LP_Interval_N_Timestamps[:5]})")
print(f"Interval N PD: {PD_Interval_N[:5]} (timestamps: {PD_Interval_N_Timestamps[:5]})")

In [ ]:

#Este es el LPPD1spkperiod
LP_Interval_xN, LP_Interval_xN_Timestamps = Interval_xN(LP_spkperiod,PD_LP_Delay,spkperiodtimestamp) #Interval N se compara con la suma de spkperiod + delay entre el último pico de LP y el primer pico de PD
PD_Interval_xN, PD_Interval_xN_Timestamps = Interval_xN(PD_spkperiod, Plateau,spkperiodtimestamp_PD) #Interval N se compara con la suma de spkperiod + delay entre el último pico de PD y el primer pico de LP

print(f"Interval N LP: {LP_Interval_xN[:5]} (timestamps: {LP_Interval_xN_Timestamps[:5]})")
print(f"Interval N PD: {PD_Interval_xN[:5]} (timestamps: {PD_Interval_xN_Timestamps[:5]})")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

min_len = min(len(LP_fst2fst), len(PD_fst2fst), len(LP_spkperiod), 
              len(PD_spkperiod), len(Plateau), len(PD_LP_Delay), len(LP_Interval_N), len(PD_Interval_N))

data = {
    'fst2fstLP': LP_fst2fst[:min_len],
    'fst2fstPD': PD_fst2fst[:min_len],
    'spkperiodLP': LP_spkperiod[:min_len],
    'spkperiodPD': PD_spkperiod[:min_len],
    'Plateau': Plateau[:min_len],
    'Delay_PD_to_LP': PD_LP_Delay[:min_len],
    'Interval_N_LP': LP_Interval_N[:min_len],
    'Interval_N_PD': PD_Interval_N[:min_len],
    'LPPD1spkperiod': LP_Interval_xN[:min_len],
    'LPPDlspkperiod': PD_Interval_xN[:min_len],
    'Time': periodotimestamp[:min_len]  # O el timestamp que prefieras
}

df = pd.DataFrame(data)

# Crear categorías de tiempo
def categorizar_tiempo(t):
    if t < 50000:
        return '0-20k s'
    elif t < 800000:
        return '20-40k s'
    else:
        return '>40k s'

df['Time_Category'] = df['Time'].apply(categorizar_tiempo)

# Pairplot con colores por categoría temporal
sns.pairplot(df, hue='Time_Category', palette=['blue', 'orange', 'red'])
plt.show()


## Contruccion de intervalos temporales bsados en hiperpolarizaciones como referencias


In [ ]:
# ==========================================
# 1. PERIODOS
# ==========================================

def periodo_hiper(ind_minimo, dt):
    """Calcula el periodo de las hiperpolarizaciones y guarda timestamps"""
    periodos_minimos = []
    timestamps = []
    for i in range(1, len(ind_minimo)):
        periodo = (ind_minimo[i] - ind_minimo[i-1]) * dt
        periodos_minimos.append(periodo)
        timestamps.append(ind_minimo[i-1] * dt)
    return periodos_minimos, timestamps

# ==========================================
# 2. INTERVALOS INTRANEURONALES (Puros)
# ==========================================


def hiper_to_Spikes_Intra(ind_minimo, ind_picos1, ind_picos2, dt):
    """Calcula tiempo desde mínimo hasta su primer y último pico DEL MISMO CICLO."""
    t_to_first = []
    t_to_last = []
    timestamps = []
    j = 0
    
    for i in range(len(ind_minimo)):
        t_min = ind_minimo[i]
        # Límite superior: el siguiente ciclo (para evitar medir ciclos silenciosos)
        t_min_sig = ind_minimo[i + 1] if i + 1 < len(ind_minimo) else float('inf')

        while j < len(ind_picos1) and ind_picos1[j] <= t_min:
            j += 1

        # Si hay un pico y ocurre ANTES del siguiente ciclo
        if j < len(ind_picos1) and ind_picos1[j] < t_min_sig:
            t_to_first.append((ind_picos1[j] - t_min) * dt)
            t_to_last.append((ind_picos2[j] - t_min) * dt)
            timestamps.append(t_min * dt)
            j += 1 
            
    return t_to_first, t_to_last, timestamps

def Spikes_to_hiper_Intra(ind_picos1, ind_picos2, ind_minimo, dt):
    """
    Calcula tiempo desde primer y último pico de cada ráfaga 
    hasta el SIGUIENTE mínimo ESTRICTAMENTE posterior al último pico.
    Sin regla de consumo: cada ráfaga busca independientemente.
    """
    t_first_to_hiper = []
    t_last_to_hiper  = []
    timestamps_first = []
    timestamps_last  = []

    for i in range(len(ind_picos1)):
        t_first = ind_picos1[i]
        t_last  = ind_picos2[i]

        # Buscar el primer mínimo ESTRICTAMENTE posterior al ÚLTIMO pico
        # (no al primero, porque la ráfaga aún no ha terminado)
        candidatos = ind_minimo[ind_minimo > t_last]

        if len(candidatos) == 0:
            continue  # Esta ráfaga es la última y no tiene hiper posterior

        t_hiper = candidatos[0]  # El más cercano posterior al último pico

        t_first_to_hiper.append((t_hiper - t_first) * dt)
        t_last_to_hiper.append((t_hiper - t_last)   * dt)
        timestamps_first.append(t_first * dt)
        timestamps_last.append(t_last   * dt)

    return t_first_to_hiper, t_last_to_hiper, timestamps_first, timestamps_last

# ==========================================
# 3. INTERVALOS INTERNEURONALES (Compuestos)
# ==========================================



def FirstSpike_to_hiper_Inter(ind_picos1, ind_minimo_n2, dt):
    """
    Primer pico N1 → siguiente hiperpolarización N2 
    estrictamente posterior al primer pico.
    Sin regla de consumo.
    """
    t_to_hiper = []
    timestamps  = []

    for t_first in ind_picos1:
        candidatos = ind_minimo_n2[ind_minimo_n2 > t_first]

        if len(candidatos) == 0:
            continue

        t_to_hiper.append((candidatos[0] - t_first) * dt)
        timestamps.append(t_first * dt)

    return t_to_hiper, timestamps


def LastSpike_to_hiper_Inter(ind_picos2, ind_minimo_n2, dt):
    """
    Último pico N1 → siguiente hiperpolarización N2 
    estrictamente posterior al último pico.
    Sin regla de consumo.
    """
    t_to_hiper = []
    timestamps  = []

    for t_last in ind_picos2:
        candidatos = ind_minimo_n2[ind_minimo_n2 > t_last]

        if len(candidatos) == 0:
            continue

        t_to_hiper.append((candidatos[0] - t_last) * dt)
        timestamps.append(t_last * dt)

    return t_to_hiper, timestamps

def hiperN1_to_N2(ind_minimo_n1, ind_minimo_n2, dt, tolerancia_max_ms=1800.0, tolerancia_min_ms=50.0):
    """
    Desfase de fase: desde mínimo N1 hasta el siguiente mínimo N2.
    Búsqueda vectorizada sin regla de consumo.
    tolerancia_max_ms: descarta saltos que superen un periodo máximo.
    tolerancia_min_ms: descarta emparejamientos físicamente imposibles (< tolerancia).
    """
    tiempos    = []
    timestamps = []

    ind_minimo_n2  = np.asarray(ind_minimo_n2)
    tolerancia_max_idx = int(tolerancia_max_ms / dt)
    tolerancia_min_idx = int(tolerancia_min_ms / dt)

    for t_n1 in ind_minimo_n1:

        candidatos = ind_minimo_n2[ind_minimo_n2 > (t_n1 + tolerancia_min_idx)]

        if len(candidatos) == 0:
            continue

        t_n2 = candidatos[0]

        if (t_n2 - t_n1) > tolerancia_max_idx:
            continue

        tiempos.append((t_n2 - t_n1) * dt)
        timestamps.append(t_n1 * dt)

    return tiempos, timestamps

def hiperN1_to_SpikesN2(ind_minimo_n1, ind_picos1_n2, ind_picos2_n2, dt, 
                         tolerancia_max_ms=1800.0, tolerancia_min_ms=50.0):
    """
    Tiempo desde mínimo N1 hasta la siguiente ráfaga N2 (primer y último pico).
    - Búsqueda vectorizada sin regla de consumo.
    - tolerancia_max_ms: descarta saltos que superen un periodo máximo.
    - tolerancia_min_ms: descarta emparejamientos físicamente imposibles.
    """
    t_hiper_to_first = []
    t_hiper_to_last  = []
    timestamps       = []

    ind_picos1_n2      = np.asarray(ind_picos1_n2)
    ind_picos2_n2      = np.asarray(ind_picos2_n2)
    tolerancia_max_idx = int(tolerancia_max_ms / dt)
    tolerancia_min_idx = int(tolerancia_min_ms / dt)

    for t_min in ind_minimo_n1:

        # Candidatos: primer pico posterior a t_min + tolerancia mínima
        mask = ind_picos1_n2 > (t_min + tolerancia_min_idx)
        if not np.any(mask):
            continue

        idx_rafaga = np.where(mask)[0][0]
        t_first    = ind_picos1_n2[idx_rafaga]
        t_last     = ind_picos2_n2[idx_rafaga]

        if (t_first - t_min) > tolerancia_max_idx:
            continue

        t_hiper_to_first.append((t_first - t_min) * dt)
        t_hiper_to_last.append( (t_last  - t_min) * dt)
        timestamps.append(t_min * dt)

    return t_hiper_to_first, t_hiper_to_last, timestamps

def SpikesN1_to_hiperN2(ind_picos1, ind_picos2, ind_minimo_n2, dt, tolerancia_min_ms=50.0):
    """
    Versión cruzada: hiper N2 estrictamente posterior al último pico N1.
    tolerancia_min_ms: descarta emparejamientos donde el intervalo es 
                       físicamente imposible (< tolerancia).
    """
    t_first_to_hiper = []
    t_last_to_hiper  = []
    timestamps_first = []
    timestamps_last  = []

    ind_minimo_n2  = np.asarray(ind_minimo_n2)
    tolerancia_idx = int(tolerancia_min_ms / dt)  # convertir ms a muestras

    for i in range(len(ind_picos1)):
        t_first = ind_picos1[i]
        t_last  = ind_picos2[i]

        # Mínimo debe ser posterior al último pico + tolerancia mínima
        candidatos = ind_minimo_n2[ind_minimo_n2 > (t_last + tolerancia_idx)]

        if len(candidatos) == 0:
            continue

        t_hiper = candidatos[0]

        t_first_to_hiper.append((t_hiper - t_first) * dt)
        t_last_to_hiper.append( (t_hiper - t_last)  * dt)
        timestamps_first.append(t_first * dt)
        timestamps_last.append( t_last  * dt)

    return t_first_to_hiper, t_last_to_hiper, timestamps_first, timestamps_last

In [ ]:
# ==============================================================================
# EJECUCIÓN DE LAS FUNCIONES Y ASIGNACIÓN DE VARIABLES
# ==============================================================================

#  Periodos Hlp y Hpd
periodos_hiper_n1, t_periodos_hiper_n1 = periodo_hiper(hiper_indices_LP, dt)
periodos_hiper_n2, t_periodos_hiper_n2 = periodo_hiper(hiper_indices_PD, dt)

#  Intervalos Intraneuronales Puros (Hiper -> Spikes)
hiper_to_firstspike_n1, hiper_to_lastspike_n1, t_hiper_n1 = hiper_to_Spikes_Intra(
    hiper_indices_LP, fSp_indices_LP, lSp_indices_LP, dt
)
hiper_to_firstspike_n2, hiper_to_lastspike_n2, t_hiper_n2 = hiper_to_Spikes_Intra(
    hiper_indices_PD, fSp_indices_PD, lSp_indices_PD, dt
)
# Los timestamps son los mismos para first y last (mismo ciclo)
t_hiper_to_lastspike_n1 = t_hiper_n1
t_hiper_to_lastspike_n2 = t_hiper_n2

#  Intervalos Intraneuronales Compuestos (Spikes -> Hiper misma neurona)

firstspike_hiper_n1, lastspike_hiper_n1, t_firstspike_hiper_n1, t_lastspike_hiper_n1 = Spikes_to_hiper_Intra(
    fSp_indices_LP, lSp_indices_LP, hiper_indices_LP, dt
)
firstspike_hiper_n2, lastspike_hiper_n2, t_firstspike_hiper_n2, t_lastspike_hiper_n2 = Spikes_to_hiper_Intra(
    fSp_indices_PD, lSp_indices_PD, hiper_indices_PD, dt
)

# Desfases Base (Hiper -> Hiper cruzado)
hiperN1_to_N2_times, t_hiperN1_to_N2_times = hiperN1_to_N2(hiper_indices_LP, hiper_indices_PD, dt)
hiperN2_to_N1_times, t_hiperN2_to_N1_times = hiperN1_to_N2(hiper_indices_PD, hiper_indices_LP, dt)

# Intervalos Compuestos Cruzados (Spikes -> Hiper cruzado)
# 
firstspikeN1_to_hiperN2_times, lastspikeN1_to_hiperN2_times, \
t_firstspikeN1_to_hiperN2_times, t_lastspikeN1_to_hiperN2_times = SpikesN1_to_hiperN2(
    fSp_indices_LP, lSp_indices_LP, hiper_indices_PD, dt
)
firstspikeN2_to_hiperN1_times, lastspikeN2_to_hiperN1_times, \
t_firstspikeN2_to_hiperN1_times, t_lastspikeN2_to_hiperN1_times = SpikesN1_to_hiperN2(
    fSp_indices_PD, lSp_indices_PD, hiper_indices_LP, dt
)

# Intervalos Compuestos Cruzados (Hiper -> Spikes cruzado)

hiperN1_to_FirstSpikeN2_times, hiperN1_to_LastSpikeN2_times, \
t_hiperN1_to_SpikesN2_times = hiperN1_to_SpikesN2(
    hiper_indices_LP, fSp_indices_PD, lSp_indices_PD, dt
)
t_hiperN1_to_FirstSpikeN2_times = t_hiperN1_to_SpikesN2_times
t_hiperN1_to_LastSpikeN2_times  = t_hiperN1_to_SpikesN2_times

hiperN2_to_FirstSpikeN1_times, hiperN2_to_LastSpikeN1_times, \
t_hiperN2_to_SpikesN1_times = hiperN1_to_SpikesN2(
    hiper_indices_PD, fSp_indices_LP, lSp_indices_LP, dt
)
t_hiperN2_to_FirstSpikeN1_times = t_hiperN2_to_SpikesN1_times
t_hiperN2_to_LastSpikeN1_times  = t_hiperN2_to_SpikesN1_times


# VERIFICACIÓN DE LONGITUDES
print("── Intraneuronales ──────────────────────────────────────")
print(f"hiper→fSp LP : {len(hiper_to_firstspike_n1):4d}  |  hiper→lSp LP : {len(hiper_to_lastspike_n1):4d}")
print(f"hiper→fSp PD : {len(hiper_to_firstspike_n2):4d}  |  hiper→lSp PD : {len(hiper_to_lastspike_n2):4d}")
print(f"fSp→hiper LP : {len(firstspike_hiper_n1):4d}  |  lSp→hiper LP : {len(lastspike_hiper_n1):4d}")
print(f"fSp→hiper PD : {len(firstspike_hiper_n2):4d}  |  lSp→hiper PD : {len(lastspike_hiper_n2):4d}")
print("── Interneuronales ──────────────────────────────────────")
print(f"HlpHpd       : {len(hiperN1_to_N2_times):4d}  |  HpdHlp       : {len(hiperN2_to_N1_times):4d}")
print(f"fSpLP→hPD    : {len(firstspikeN1_to_hiperN2_times):4d}  |  lSpLP→hPD    : {len(lastspikeN1_to_hiperN2_times):4d}")
print(f"fSpPD→hLP    : {len(firstspikeN2_to_hiperN1_times):4d}  |  lSpPD→hLP    : {len(lastspikeN2_to_hiperN1_times):4d}")
print(f"hLP→fSpPD    : {len(hiperN1_to_FirstSpikeN2_times):4d}  |  hLP→lSpPD    : {len(hiperN1_to_LastSpikeN2_times):4d}")
print(f"hPD→fSpLP    : {len(hiperN2_to_FirstSpikeN1_times):4d}  |  hPD→lSpLP    : {len(hiperN2_to_LastSpikeN1_times):4d}")

#### Unirlos en un dataframe

In [ ]:
import pandas as pd

df_hiper = pd.DataFrame({
    'Time': t_firstspike_hiper_n1,
})

df_hiper = df_hiper.sort_values('Time')

datos_a_unir = [
    ("Periodo Hlp(ms)",        periodos_hiper_n1,              t_periodos_hiper_n1),
    ("Periodo Hpd(ms)",        periodos_hiper_n2,              t_periodos_hiper_n2),
    # Intraneuronales Hiper→Spikes  (timestamp = t_hiper_n1/n2)
    ('Intervalo HlpFSlp(ms)',  hiper_to_firstspike_n1,         t_hiper_n1),
    ('Intervalo HlpLSlp(ms)',  hiper_to_lastspike_n1,          t_hiper_n1),
        ('Intervalo HpdFSpd(ms)',  hiper_to_firstspike_n2,         t_hiper_n2),
        ('Intervalo HpdLSpd(ms)',  hiper_to_lastspike_n2,          t_hiper_n2),
        # Intraneuronales Spikes→Hiper  (timestamps separados)
        ('Intervalo FSlpHlp(ms)',  firstspike_hiper_n1,            t_firstspike_hiper_n1),
        ('Intervalo LSlpHlp(ms)',  lastspike_hiper_n1,             t_lastspike_hiper_n1),
        ('Intervalo FSpdHpd(ms)',  firstspike_hiper_n2,            t_firstspike_hiper_n2),
        ('Intervalo LSpdHpd(ms)',  lastspike_hiper_n2,             t_lastspike_hiper_n2),
        # Hiper→Hiper cruzado
        ('Intervalo HlpHpd(ms)',   hiperN1_to_N2_times,            t_hiperN1_to_N2_times),
        ('Intervalo HpdHlp(ms)',   hiperN2_to_N1_times,            t_hiperN2_to_N1_times),
        # Spikes→Hiper cruzado (timestamps separados por first/last)
        ('Intervalo FSlpHpd(ms)',  firstspikeN1_to_hiperN2_times,  t_firstspikeN1_to_hiperN2_times),
        ('Intervalo LSlpHpd(ms)',  lastspikeN1_to_hiperN2_times,   t_lastspikeN1_to_hiperN2_times),
        ('Intervalo FSpdHlp(ms)',  firstspikeN2_to_hiperN1_times,  t_firstspikeN2_to_hiperN1_times),
        ('Intervalo LSpdHlp(ms)',  lastspikeN2_to_hiperN1_times,   t_lastspikeN2_to_hiperN1_times),
        # Hiper→Spikes cruzado (timestamp compartido)
        ('Intervalo HlpFSpd(ms)',  hiperN1_to_FirstSpikeN2_times,  t_hiperN1_to_FirstSpikeN2_times),
        ('Intervalo HlpLSpd(ms)',  hiperN1_to_LastSpikeN2_times,   t_hiperN1_to_LastSpikeN2_times),
        ('Intervalo HpdFSlp(ms)',  hiperN2_to_FirstSpikeN1_times,  t_hiperN2_to_FirstSpikeN1_times),
        ('Intervalo HpdLSlp(ms)',  hiperN2_to_LastSpikeN1_times,   t_hiperN2_to_LastSpikeN1_times),
    ]


In [ ]:
import numpy as np

def revisar_limites_datos(datos_a_unir):
    print("="*75)
    print(f"{'NOMBRE DEL INTERVALO':<25} | {'MIN (ms)':<10} | {'MAX (ms)':<10} | {'MEDIA (ms)':<10} | {'Nº DATOS':<10}")
    print("="*75)
    
    for nombre, lista_valores, _ in datos_a_unir:
       
        vals = np.array(lista_valores)
        vals = vals[~np.isnan(vals)] 
        
        if len(vals) > 0:
            val_min = np.min(vals)
            val_max = np.max(vals)
            val_mean = np.mean(vals)
            num_datos = len(vals)
            
            
            print(f"{nombre:<25} | {val_min:<10.2f} | {val_max:<10.2f} | {val_mean:<10.2f} | {num_datos:<10}")
            
           
            if val_min <= 0:
                print(f"Hay valores <= 0 en {nombre}.")
        else:
            print(f"{nombre:<25} | {'VACÍO':<10} | {'VACÍO':<10} | {'VACÍO':<10} | {0:<10}")
            print(f" La lista de {nombre} está vacía.")
            
    print("="*75)

revisar_limites_datos(datos_a_unir)

In [ ]:

for nombre_col, valores, tiempos in datos_a_unir:
    
    df_temp = pd.DataFrame({
        'Time': tiempos,
        nombre_col: valores
    }).sort_values('Time')
    

    df_hiper = pd.merge_asof(
        df_hiper,
        df_temp,
        on='Time',
        direction='nearest',
        tolerance=5000
    )

df_lp = pd.DataFrame({
    'Periodo LP(ms)':               LP_fst2fst,
    'Intervalo LPPD1spkperiod(ms)': LP_Interval_N,
}).reset_index(drop=True)

df_hiper = df_hiper.reset_index(drop=True)
df_hiper = pd.concat([df_hiper, df_lp], axis=1)


df_hiper = df_hiper.dropna()


bins   = [0, 400000, 800000, np.inf]
labels = ['0-400k ms', '400k-800k ms', '>800k ms']
df_hiper['Time_Category'] = pd.cut(df_hiper['Time'], bins=bins, labels=labels)


# Pairplot con TODOS los intervalos
g = sns.pairplot(
    df_hiper, 
    hue='Time_Category', 
    palette=['blue', 'orange', 'red'],
    plot_kws={'alpha': 0.5, 's': 15},  # Puntos más pequeños y transparentes
    diag_kind='kde',  # KDE en diagonal para evitar saturación
    corner=False  # Mostrar matriz completa
)

g.fig.suptitle('Pairplot Completo: Todos los Intervalos', y=1.01, fontsize=16)
plt.tight_layout()
plt.show()

#### Pairplot con R^2

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

columnas_a_graficar = [col for col in df_hiper.columns if col not in ['Time', 'Time_Category']]

sns.set_style("white")  # ← sin grid desde el principio

g = sns.pairplot(
    df_hiper,
    vars=columnas_a_graficar,
    hue='Time_Category',
    palette='turbo',
    plot_kws={'alpha': 0.5, 's': 20, 'edgecolor': 'none'},
    diag_kind='kde'
)

# ← por si acaso quedan residuos de grid
for ax_row in g.axes:
    for ax in ax_row:
        if ax is not None:
            ax.grid(False)

corr_matrix = df_hiper[columnas_a_graficar].corr()
r2_matrix   = corr_matrix ** 2
cmap        = plt.get_cmap('Reds')

for i, row in enumerate(g.axes):
    for j, ax in enumerate(row):
        if ax is None:
            continue
        if j > i:
            ax.set_visible(True)
            for artist in ax.lines + ax.collections + ax.patches:
                artist.set_visible(False)

            var_y = g.y_vars[i]
            var_x = g.x_vars[j]

            try:
                r2 = r2_matrix.loc[var_y, var_x]
                if np.isnan(r2): r2 = 0.0
            except KeyError:
                r2 = 0.0

            bg_color   = cmap(r2)
            text_color = 'white' if r2 > 0.5 else 'black'

            ax.set_facecolor(bg_color)
            ax.tick_params(left=False, bottom=False,
                           labelleft=False, labelbottom=False)

            for spine in ax.spines.values():
                spine.set_visible(False)

            ax.annotate(f"{r2:.4f}",
                        xy=(0.5, 0.5), xycoords='axes fraction',
                        ha='center', va='center',
                        fontsize=30, fontweight='bold', color=text_color)

n = len(columnas_a_graficar)
for i in range(n):
    g.axes[i][0].tick_params(labelleft=True)
    g.axes[n-1][i].tick_params(labelbottom=True)

sns.move_legend(
    g,
    "center right",
    bbox_to_anchor=(0.98, 0.5),
    title='Fase Temporal',
    frameon=True
)

# Texto más grande
plt.setp(g._legend.get_title(), fontsize=18, weight='bold')
plt.setp(g._legend.get_texts(), fontsize=15)

# Marcadores más grandes
for handle in g._legend.legend_handles:
    try:
        handle.set_sizes([260])
    except AttributeError:
        handle.set_markersize(22)

g.figure.subplots_adjust(right=0.95, hspace=0.3, wspace=0.3)
plt.show()

#### Plots seleccionados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


relaciones = [
    ('Intervalo HpdLSpd(ms)', 'Intervalo HpdFSpd(ms)', 'Invariante'),
    ('Intervalo FSpdHpd(ms)', 'Intervalo LSpdHpd(ms)', 'Invariante'),
    ('Periodo Hlp(ms)', 'Intervalo LSlpHlp(ms)', 'Invariante'), 
    ('Periodo Hlp(ms)', 'Intervalo HlpFSpd(ms)', 'Invariante'),
    ('Periodo Hlp(ms)', 'Intervalo FSpdHpd(ms)', 'Variante'),
    ('Intervalo HpdHlp(ms)', 'Intervalo HlpFSlp(ms)', 'Variante'),
    ("Periodo LP(ms)", 'Intervalo LPPD1spkperiod(ms)', 'Invariante')
]

sns.set_theme(style="whitegrid")

for x_col, y_col, tipo in relaciones:

    g = sns.jointplot(
        data=df_hiper,
        x=x_col,
        y=y_col,
        hue='Time_Category',
        palette='turbo',
        alpha=0.5,      
        s=25,            
        marginal_kws={'fill': True}
    )
    
    g.set_axis_labels(x_col, y_col, fontsize=12)

    ax = g.ax_joint
    ax.margins(x=0.05, y=0.05) 
    

    if ax.get_legend() is not None:
        ax.get_legend().remove()
    
    plt.show() 

In [ ]:
print(df_hiper.columns)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


corr = df_hiper.drop(columns=['Time', 'Time_Category']).corr()

plt.figure(figsize=(15, 10))
sns.heatmap(corr, annot=False, cmap='gray', center=0)
plt.title("¿Qué intervalos están relacionados realmente?")
plt.show()

In [ ]:


r2 = corr ** 2
plt.figure(figsize=(15, 10))
sns.heatmap(r2, 
            annot=False, 
            cmap='Spectral_r', 
            center=0.5)        
plt.title("Matriz de R² con Mapa Espectral")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr

# Reutilizamos tu diccionario para que el resultado salga limpio
nombres_ejes = {
    'Periodo H1(ms)': 'Periodo Hlp (ms)',
    'Periodo H2(ms)': 'Periodo Hpd (ms)',
    'Intervalo FS1H1(ms)': 'Intervalo FSlpHlp (ms)',
    'Intervalo FS2H2(ms)': 'Intervalo FSpdHpd (ms)',
    'Intervalo LS2H2(ms)': 'Intervalo LSpdHpd (ms)',
    'Intervalo LS2H1(ms)': 'Intervalo LSpdHlp (ms)',
    'Intervalo H1FS2(ms)': 'Intervalo HlpFSpd (ms)',
    'Intervalo H1H2(ms)': 'Intervalo HlpHpd (ms)',
    'Intervalo H2H1(ms)': 'Intervalo HpdHlp (ms)',
    'Intervalo FS1H2(ms)': 'Intervalo FSlpHpd (ms)',
    'Intervalo FS2H1(ms)': 'Intervalo FSpdHlp (ms)',
    'Intervalo LS1H2(ms)': 'Intervalo LSlpHpd (ms)',
    'Intervalo H2FS1(ms)': 'Intervalo HpdFSpd (ms)',
    'Intervalo H1LS2(ms)': 'Intervalo HlpLSpd (ms)',
    'Intervalo H2LS1(ms)': 'Intervalo HpdLSlp (ms)',
}

# Obtener columnas numéricas (excluyendo metadatos)
cols = df_hiper.drop(columns=['Time', 'Time_Category'], errors='ignore').columns

resultados = []

# Iterar por el triángulo superior sin duplicados
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        col1 = cols[i]
        col2 = cols[j]
        
        # Eliminar NaNs para calcular la correlación limpia
        df_clean = df_hiper[[col1, col2]].dropna()
        data1 = df_clean[col1]
        data2 = df_clean[col2]
        
        if len(data1) > 2: # Necesitamos datos suficientes
            r, p_val = pearsonr(data1, data2)
            r2 = r ** 2
            
            # Clasificación de la fuerza de la relación
            if r2 >= 0.7:
                fuerza = "Fuerte"
            elif r2 >= 0.3:
                fuerza = "Moderada"
            else:
                fuerza = "Débil"
                
            # Dirección de la tendencia
            direccion = "Directa (+)" if r > 0 else "Inversa (-)"
                
            resultados.append({
                'Variable 1': nombres_ejes.get(col1, col1),
                'Variable 2': nombres_ejes.get(col2, col2),
                'R2': round(r2, 4),
                'r (Pearson)': round(r, 4),
                'Dirección': direccion,
                'Fuerza': fuerza,
                'p-valor': p_val
            })

df_resultados = pd.DataFrame(resultados).sort_values(by='R2', ascending=False)


print(" RELACIONES FUERTES (Invariantes Estructurales) ")
fuertes = df_resultados[df_resultados['Fuerza'] == 'Fuerte']
print(fuertes[['Variable 1', 'Variable 2', 'R2', 'Dirección']].to_string(index=False) if not fuertes.empty else "Ninguna")

print("\n RELACIONES MODERADAS (Parcialmente acopladas) ")
moderadas = df_resultados[df_resultados['Fuerza'] == 'Moderada']
print(moderadas[['Variable 1', 'Variable 2', 'R2', 'Dirección']].head(10).to_string(index=False) if not moderadas.empty else "Ninguna")

print("\n PARES COMPLETAMENTE INDEPENDIENTES (R² cercano a 0) ")
# Mostramos las 5 con el R2 más bajo
debiles = df_resultados.tail(5)
print(debiles[['Variable 1', 'Variable 2', 'R2']].to_string(index=False))

In [ ]:

print("\n--- RECUENTO TOTAL ---")
print(df_resultados['Fuerza'].value_counts())

# Pairplot completo entre hiperopolarizaciones y potenciales de acción

In [ ]:

# Encontrar la longitud mínima
min_len = min(
    len(periodos_hiper_n1), len(periodos_hiper_n2),
    len(periodos_hiper_n1), len(periodos_hiper_n2),
    len(firstspike_hiper_n1), len(firstspike_hiper_n2),
    len(lastspike_hiper_n1), len(lastspike_hiper_n2),
    len(hiperN2_to_N1_times), len(hiperN1_to_N2_times),
    len(firstspikeN1_to_hiperN2_times), len(firstspikeN2_to_hiperN1_times),
    len(lastspikeN1_to_hiperN2_times), len(lastspikeN2_to_hiperN1_times),
    len(hiperN1_to_FirstSpikeN2_times), len(hiperN2_to_FirstSpikeN1_times),
    len(hiperN1_to_LastSpikeN2_times), len(hiperN2_to_LastSpikeN1_times),
    len(LP_fst2fst), len(PD_fst2fst), len(LP_spkperiod), 
              len(PD_spkperiod), len(Plateau), len(PD_LP_Delay), len(LP_Interval_N), len(PD_Interval_N)
)

# Crear diccionario con TODOS los datos
data_completo = {
    
    # Intervalos de Hiperpolarizaciones
    'Periodo H1(s)': periodos_hiper_n1[:min_len],
    'Periodo H2(s)': periodos_hiper_n2[:min_len],
    'Intervalo FS1H1(s)': firstspike_hiper_n1[:min_len],
    'Intervalo FS2H2(s)': firstspike_hiper_n2[:min_len],
    'Intervalo LS1H1(s)': lastspike_hiper_n1[:min_len],
    'Intervalo LS2H2(s)': lastspike_hiper_n2[:min_len],
    'Intervalo H2H1(s)': hiperN2_to_N1_times[:min_len],
    'Intervalo H1H2(s)': hiperN1_to_N2_times[:min_len],
    'Intervalo FS1H2(s)': firstspikeN1_to_hiperN2_times[:min_len],
    'Intervalo FS2H1(s)': firstspikeN2_to_hiperN1_times[:min_len],
    'Intervalo LS1H2(s)': lastspikeN1_to_hiperN2_times[:min_len],
    'Intervalo LS2H1(s)': lastspikeN2_to_hiperN1_times[:min_len],
    'Intervalo H1FS2(s)': hiperN1_to_FirstSpikeN2_times[:min_len],
    'Intervalo H2FS1(s)': hiperN2_to_FirstSpikeN1_times[:min_len],
    'Intervalo H1LS2(s)': hiperN1_to_LastSpikeN2_times[:min_len],
    'Intervalo H2LS1(s)': hiperN2_to_LastSpikeN1_times[:min_len],
    'fst2fstLP': LP_fst2fst[:min_len],
    'fst2fstPD': PD_fst2fst[:min_len],
    'spkperiodLP': LP_spkperiod[:min_len],
    'spkperiodPD': PD_spkperiod[:min_len],
    'Plateau': Plateau[:min_len],
    'Delay_PD_to_LP': PD_LP_Delay[:min_len],
    'LPPD1spkperiod': LP_Interval_N[:min_len],
    'Interval_N_PD': PD_Interval_N[:min_len],
    'Interval_xN_LP': LP_Interval_xN[:min_len],
    'LPPDlspkperiod': PD_Interval_xN[:min_len],
    # Timestamp de referencia
    'Time': t_periodos_hiper_n1[:min_len]
}

df_completo = pd.DataFrame(data_completo)

# Categorizar por tiempo
bins = [0, 400000, 800000, np.inf]
labels = ['0-400k s', '400k-800k s', '>800k s']
df_completo['Time_Category'] = pd.cut(df_completo['Time'], bins=bins, labels=labels)

# ============================================================
# PAIRPLOT COMPLETO
# ============================================================

# Pairplot con TODOS los intervalos
g = sns.pairplot(
    df_completo, 
    hue='Time_Category', 
    palette=['blue', 'orange', 'red'],
    plot_kws={'alpha': 0.5, 's': 15},  # Puntos más pequeños y transparentes
    diag_kind='kde',  # KDE en diagonal para evitar saturación
    corner=False  # Mostrar matriz completa
)

g.fig.suptitle('Pairplot Completo: Todos los Intervalos', y=1.01, fontsize=16)
plt.tight_layout()
plt.show()



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calcula la matriz de correlación
corr = df_completo.drop(columns=['Time', 'Time_Category']).corr()

plt.figure(figsize=(15, 10))
sns.heatmap(corr, annot=False, cmap='gray', center=0)
plt.title("¿Qué intervalos están relacionados realmente?")
plt.show()

In [ ]:
# Opción B: Mapa espectral para máximo contraste visual
plt.figure(figsize=(15, 10))
sns.heatmap(r2, 
            annot=False, 
            cmap='Spectral_r', # El rojo será el 1.0 y el azul el 0.0
            center=0.5)        # Centra el color neutro en 0.5
plt.title("Matriz de R² con Mapa Espectral")
plt.show()

In [ ]:
import numpy as np

# 1. Partimos de la matriz de correlación
corr = df_completo.drop(columns=['Time', 'Time_Category']).corr()

# 2. Convertimos a R²
r2 = corr ** 2

# 3. Nos quedamos solo con el triángulo superior (sin duplicados ni diagonal)
mask = np.triu(np.ones_like(r2, dtype=bool), k=1)
r2_upper = r2.where(mask)

# 4. Convertimos a lista ordenada
r2_pairs = (
    r2_upper
    .unstack()
    .dropna()
    .sort_values(ascending=False)
)

# 5. (Opcional) filtrar relaciones interesantes
r2_filtered = r2_pairs[(r2_pairs > 0.7) & (r2_pairs < 0.99)]

# 6. Mostrar resultados
print("Top 20 relaciones más altas:")
for (var1, var2), value in r2_pairs.head(20).items():
    print(f"{var1}  vs  {var2}  →  R² = {value:.3f}")

print("\nRelaciones fuertes (0.7 < R² < 0.99):")
for (var1, var2), value in r2_filtered.items():
    print(f"{var1}  vs  {var2}  →  R² = {value:.3f}")

# Graficas Boxplots


In [ ]:
import matplotlib.patches as mpatches
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Reutilizamos tu diccionario de traducción
nombres_ejes = {
    'Periodo H1(ms)': 'Periodo Hlp (ms)',
    'Periodo H2(ms)': 'Periodo Hpd (ms)',
    'Intervalo FS1H1(ms)': 'Intervalo FSlpHlp (ms)',
    'Intervalo FS2H2(ms)': 'Intervalo FSpdHpd (ms)',
    'Intervalo LS2H2(ms)': 'Intervalo LSpdHpd (ms)',
    'Intervalo LS2H1(ms)': 'Intervalo LSpdHlp (ms)',
    'Intervalo H1FS2(ms)': 'Intervalo HlpFSpd (ms)',
    'Intervalo H1H2(ms)': 'Intervalo HlpHpd (ms)',
    'Intervalo H2H1(ms)': 'Intervalo HpdHlp (ms)',
    'Intervalo FS1H2(ms)': 'Intervalo FSlpHpd (ms)',
    'Intervalo FS2H1(ms)': 'Intervalo FSpdHlp (ms)',
    'Intervalo LS1H2(ms)': 'Intervalo LSlpHpd (ms)',
    'Intervalo LS2H1(ms)': 'Intervalo LSpdHlp (ms)',
    'Intervalo H1FS2(ms)': 'Intervalo HlpFSpd (ms)',
    'Intervalo H2FS1(ms)': 'Intervalo HpdFSpd (ms)',
    'Intervalo H1LS2(ms)': 'Intervalo HlpLSpd (ms)',
    'Intervalo H2LS1(ms)': 'Intervalo HpdLSlp (ms)',
}

columnas_intervalos = [col for col in df_hiper.columns if col not in ['Time', 'Time_Category']]

graficas_por_figura = 11

colores_totales = sns.color_palette("turbo", n_colors=len(columnas_intervalos))

for i in range(0, len(columnas_intervalos), graficas_por_figura):
    bloque_columnas = columnas_intervalos[i : i + graficas_por_figura]
    colores_bloque = colores_totales[i : i + graficas_por_figura]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    df_melted = pd.melt(df_hiper[bloque_columnas], var_name='Intervalo', value_name='Tiempo (s)')
    

    propiedades_atipicos = dict(marker='o', markerfacecolor='black', markersize=4, alpha=0.6, linestyle='none')

    bplot = sns.boxplot(
        data=df_melted,
        x='Intervalo',
        y='Tiempo (s)',
        hue='Intervalo',          
        palette=colores_bloque,   
        ax=ax,
        width=0.5,
        linewidth=1.5,
        dodge=False,
        flierprops=propiedades_atipicos # Aplicamos las propiedades aquí
    )
    
    print(f"\n--- ESTADÍSTICAS DEL BLOQUE {i//graficas_por_figura + 1} ---")
    
    # Calcular y añadir el CV encima de cada caja
    for j, col in enumerate(bloque_columnas):
        datos_col = df_hiper[col].dropna()
        if len(datos_col) > 0:
            media = np.mean(datos_col)
            desv_std = np.std(datos_col)
            cv_porcentaje = (desv_std / abs(media)) * 100 if media != 0 else 0
            
 
            Q1 = np.percentile(datos_col, 25)
            Q3 = np.percentile(datos_col, 75)
            IQR = Q3 - Q1
            limite_inferior = Q1 - 1.5 * IQR
            limite_superior = Q3 + 1.5 * IQR
            
   
            outliers = datos_col[(datos_col < limite_inferior) | (datos_col > limite_superior)]
            num_outliers = len(outliers)
            pct_outliers = (num_outliers / len(datos_col)) * 100
            mediana_val = np.median(datos_col)
            
           
            nombre_traducido = nombres_ejes.get(col, col)
            print(f"{nombre_traducido}: Mediana = {mediana_val:.1f} | Atípicos = {num_outliers} ({pct_outliers:.1f}%)")
            
            # Colocar texto del CV en la gráfica
            ymin, ymax = ax.get_ylim()
            y_text = ymax - (ymax - ymin) * 0.03 
            
            ax.text(
                x=j, y=y_text, s=f"CV:\n{cv_porcentaje:.1f}%",
                horizontalalignment='center', verticalalignment='top',
                fontsize=11, fontweight='bold', color=colores_bloque[j],
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2)
            )


    ax.set_xlabel('') 
    ax.set_xticks([]) 
    ax.set_ylabel('Duración de los intervalos (ms)', fontsize=12, fontweight='bold')
    

    handles = []
    for k, col_original in enumerate(bloque_columnas):
        nombre_nuevo = nombres_ejes.get(col_original, col_original)
        patch = mpatches.Patch(color=colores_bloque[k], label=nombre_nuevo)
        handles.append(patch)
    
    ax.legend(
        handles=handles, 
        title='Intervalos Temporales', 
        bbox_to_anchor=(1.01, 1), 
        loc='upper left', 
        frameon=True
    )
    
    plt.tight_layout()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()

# Verificación de los intervalos

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def graficar_un_intervalo_con_senales(datos_a_unir, nombre_intervalo, senal_LP, senal_PD, dt, num_ciclos=3, indice_inicio=10):
    """
    Grafica UN SOLO intervalo junto con las dos señales crudas para validación visual.
    Versión mejorada estéticamente.
    """
    
    data = next((item for item in datos_a_unir if item[0] == nombre_intervalo), None)

    if not data:
        print(f" No se encontró '{nombre_intervalo}' en la lista.")
        return

    _, duraciones, tiempos = data

    
    tiempo_total = np.arange(len(senal_LP)) * dt


    fig, (ax_lp, ax_pd, ax_gantt) = plt.subplots(
        3, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios': [2, 2, 1.2]}
    )
    fig.subplots_adjust(hspace=0.05) 

    # --- Graficar Señales Crudas ---
    ax_lp.plot(tiempo_total, senal_LP, color='#111111', linewidth=1.2)
    ax_lp.set_ylabel("Voltaje LP", fontweight='bold', fontsize=12)
    ax_lp.grid(True, linestyle=':', alpha=0.6)
    ax_lp.spines['top'].set_visible(False)
    ax_lp.spines['right'].set_visible(False)

    ax_pd.plot(tiempo_total, senal_PD, color='#555555', linewidth=1.2)
    ax_pd.set_ylabel("Voltaje PD", fontweight='bold', fontsize=12)
    ax_pd.grid(True, linestyle=':', alpha=0.6)
    ax_pd.spines['top'].set_visible(False)
    ax_pd.spines['right'].set_visible(False)

  
    pos_y = 1.0
    color_intervalo = 'crimson'  
    color_linea_guia = '#FF7F50' 

    x_min_zoom = float('inf')
    x_max_zoom = 0


    limite = min(indice_inicio + num_ciclos, len(tiempos))
    for i in range(indice_inicio, limite):
        t_start = tiempos[i]
        t_end = t_start + duraciones[i]
        
        x_min_zoom = min(x_min_zoom, t_start)
        x_max_zoom = max(x_max_zoom, t_end)

        # Sombra de fondo en las señales con bordes definidos
        ax_lp.axvspan(t_start, t_end, color=color_intervalo, alpha=0.1)
        ax_pd.axvspan(t_start, t_end, color=color_intervalo, alpha=0.1)

        # Líneas verticales guía que cruzan todos los gráficos
        for ax in [ax_lp, ax_pd, ax_gantt]:
            ax.axvline(t_start, color=color_linea_guia, linestyle='--', linewidth=1.5, alpha=0.8)
            ax.axvline(t_end, color=color_linea_guia, linestyle='--', linewidth=1.5, alpha=0.8)

        # Línea horizontal en el panel inferior (Gantt)
        ax_gantt.hlines(y=pos_y, xmin=t_start, xmax=t_end, color=color_intervalo, linewidth=5)
        
        # Marcadores de inicio y fin más limpios
        ax_gantt.plot(t_start, pos_y, '|', color='black', markersize=15, markeredgewidth=2)
        ax_gantt.plot(t_end, pos_y, '|', color='black', markersize=15, markeredgewidth=2)
        
        # Texto con la duración (¡Corregido el fontsize!)
        ax_gantt.text(t_start + (duraciones[i]/2), pos_y + 0.15, f"{duraciones[i]:.1f} ms", 
                      ha='center', va='bottom', fontsize=11, color='black', 
                      fontweight='bold', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

    ax_gantt.set_yticks([pos_y])
    ax_gantt.set_yticklabels([nombre_intervalo], fontsize=12, fontweight='bold', color=color_intervalo)
    ax_gantt.set_ylim(0.4, 1.6)
    
    
    ax_gantt.set_xlabel("Tiempo Absoluto (ms)", fontsize=12, fontweight='bold')
    ax_gantt.grid(True, axis='x', linestyle='--', alpha=0.7)
    ax_gantt.spines['top'].set_visible(False)
    ax_gantt.spines['right'].set_visible(False)
    ax_gantt.spines['left'].set_visible(False)

    
    if x_min_zoom != float('inf') and x_max_zoom != 0:
        margen = (x_max_zoom - x_min_zoom) * 0.15
        ax_lp.set_xlim(x_min_zoom - margen, x_max_zoom + margen)

    fig.suptitle(f"Validación Visual: {nombre_intervalo}", fontsize=18, fontweight='black', y=0.95)
    plt.show(block=True)

In [ ]:

intervalos_a_validar = [
    "Periodo Hlp(ms)",
    "Periodo Hpd(ms)",
    "Intervalo FSlpHlp(ms)",
    "Intervalo FSpdHpd(ms)",
    "Intervalo LSlpHlp(ms)",
    "Intervalo LSpdHpd(ms)",
    "Intervalo HpdHlp(ms)",
    "Intervalo HlpHpd(ms)",
    "Intervalo FSlpHpd(ms)",
    "Intervalo FSpdHlp(ms)",
    "Intervalo LSlpHpd(ms)",
    "Intervalo LSpdHlp(ms)",
    "Intervalo HlpFSpd(ms)",
    "Intervalo HpdFSlp(ms)",
    "Intervalo HlpLSpd(ms)",
    "Intervalo HpdLSlp(ms)",
    "Intervalo HlpFSlp(ms)",
    "Intervalo HpdFSpd(ms)",
    "Intervalo HlpLSlp(ms)",
    "Intervalo HpdLSpd(ms)",
    "Periodo LP(ms)",
    "Intervalo LPPD1spkperiod(ms)"
]

print("Iniciando validación visual exhaustiva con señales...")
print("Cierra la ventana del gráfico para ver el siguiente intervalo (son 20 en total).")

for intervalo in intervalos_a_validar:
    graficar_un_intervalo_con_senales(
        datos_a_unir, 
        nombre_intervalo=intervalo, 
        senal_LP=señal_LP,  
        senal_PD=señal_PD, 
        dt=dt,              
        num_ciclos=20,       
        indice_inicio=1000    
    )

print("Validación terminada.")

In [ ]:
def grafico_maestro_tfg(datos_a_unir, intervalos, senal_LP, senal_PD, dt, num_ciclos=3):
   
    fig, (ax_lp, ax_pd, ax_matrix) = plt.subplots(
        3, 1, figsize=(12, 10), sharex=True, gridspec_kw={'height_ratios': [1.5, 1.5, 6]}
    )
    fig.subplots_adjust(hspace=0.08)
    
    dt_ms = dt * 1000.0 if dt < 0.1 else dt
    tiempo_total = np.arange(len(senal_LP)) * dt_ms
    
   
    ax_lp.plot(tiempo_total, senal_LP, color='black', linewidth=0.7)
    ax_lp.set_ylabel("Voltaje LP", fontweight='bold')
    ax_pd.plot(tiempo_total, senal_PD, color='dimgray', linewidth=0.7)
    ax_pd.set_ylabel("Voltaje PD", fontweight='bold')
 
    colores = plt.cm.tab20(np.linspace(0, 1, len(intervalos)))
    
    x_min, x_max = float('inf'), 0
    
    for idx, nombre_int in enumerate(intervalos):
        data = next((item for item in datos_a_unir if item[0] == nombre_int), None)
        if not data: continue
        _, duraciones, tiempos = data
        
    
        pos_y = idx 
        
        # Graficar los ciclos asignados
        limite = min(10 + num_ciclos, len(tiempos)) 
        for i in range(10, limite):
            t_start = tiempos[i]
            t_end = t_start + duraciones[i]
            x_min, x_max = min(x_min, t_start), max(x_max, t_end)
            
            # Línea de intervalo en su respectiva fila
            ax_matrix.hlines(y=pos_y, xmin=t_start, xmax=t_end, color=colores[idx], linewidth=4)
            ax_matrix.plot(t_start, pos_y, 'o', color=colores[idx], markersize=5)
            ax_matrix.plot(t_end, pos_y, '>', color=colores[idx], markersize=5)
            
    ax_matrix.set_yticks(range(len(intervalos)))
    ax_matrix.set_yticklabels(intervalos, fontsize=9, fontweight='bold')
    ax_matrix.set_ylim(-0.5, len(intervalos) - 0.5)
    ax_matrix.invert_yaxis() # Para que el primer intervalo de la lista salga arriba
    ax_matrix.set_xlabel("Tiempo Absoluto (ms)", fontsize=11, fontweight='bold')
    

    ax_matrix.grid(True, axis='x', linestyle=':', alpha=0.6)
    ax_lp.grid(True, axis='x', linestyle=':', alpha=0.6)
    ax_pd.grid(True, axis='x', linestyle=':', alpha=0.6)
    
    if x_min != float('inf'):
        margen = (x_max - x_min) * 0.05
        ax_matrix.set_xlim(x_min - margen, x_max + margen)
        
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def graficar_gantt_multicapa(datos_a_unir, intervalos_a_validar, senal_LP, senal_PD, dt, num_ciclos=10, indice_inicio=5):
    """
    Grafica un Diagrama de Gantt multicapa sincronizado con las señales de voltaje.
    Traza una cuadrícula vertical exacta basada en los 6 eventos fundamentales.
    """
   
    ref_nombre = "Periodo Hlp(ms)" 
    data_ref = next((item for item in datos_a_unir if item[0] == ref_nombre), None)
    
    if not data_ref:
        print(f"No se encontró el intervalo de referencia '{ref_nombre}'.")
        return

    _, duraciones_ref, tiempos_ref = data_ref
    limite_ref = min(indice_inicio + num_ciclos, len(tiempos_ref) - 1)
    
    t_min = tiempos_ref[indice_inicio]
    t_max = tiempos_ref[limite_ref] + duraciones_ref[limite_ref]
    margen = (t_max - t_min) * 0.02 
    
    t_lim_inf = t_min - margen
    t_lim_sup = t_max + margen

    # 2. Reconstruir el vector de tiempo total para las señales
    tiempo_total = np.arange(len(senal_LP)) * dt

    # 3. Configurar la figura 
    fig, (ax_lp, ax_pd, ax_gantt) = plt.subplots(
        3, 1, figsize=(16, 12), sharex=True, gridspec_kw={'height_ratios': [1, 1, 4]}
    )
    fig.subplots_adjust(hspace=0.08)

    # --- Graficar Señales Crudas ---
    ax_lp.plot(tiempo_total, senal_LP, color='#111111', linewidth=1, zorder=2)
    ax_lp.set_ylabel("Voltaje LP", fontweight='bold', fontsize=11)
    ax_lp.grid(False) 
    ax_lp.spines['top'].set_visible(False)
    ax_lp.spines['right'].set_visible(False)

    ax_pd.plot(tiempo_total, senal_PD, color='#555555', linewidth=1, zorder=2)
    ax_pd.set_ylabel("Voltaje PD", fontweight='bold', fontsize=11)
    ax_pd.grid(False)
    ax_pd.spines['top'].set_visible(False)
    ax_pd.spines['right'].set_visible(False)


    eventos_base = {
        "Hlp":  "Periodo Hlp(ms)",
        "Hpd":  "Periodo Hpd(ms)",
        "FSlp": "Intervalo FSlpHlp(ms)",
        "LSlp": "Intervalo LSlpHlp(ms)",
        "FSpd": "Intervalo FSpdHpd(ms)",
        "LSpd": "Intervalo LSpdHpd(ms)"
    }


    for nombre_evento, nombre_intervalo in eventos_base.items():
        data_evento = next((item for item in datos_a_unir if item[0] == nombre_intervalo), None)
        
        if data_evento:
            _, _, tiempos_evento = data_evento
            
            es_hiper = (nombre_evento in ["Hlp", "Hpd"])
            estilo_linea = '-' if es_hiper else '--'
            color_linea = '#555555' if es_hiper else '#999999'
            grosor = 1.2 if es_hiper else 1.0
            transparencia = 0.5 if es_hiper else 0.4

            for t_start in tiempos_evento:
                if t_lim_inf <= t_start <= t_lim_sup:
                    for ax in [ax_lp, ax_pd, ax_gantt]:
                        ax.axvline(t_start, color=color_linea, linestyle=estilo_linea, 
                                   linewidth=grosor, alpha=transparencia, zorder=1)

    num_intervalos = len(intervalos_a_validar)
    colores = plt.cm.tab20(np.linspace(0, 1, num_intervalos))

    for i in range(num_intervalos):
        y_pos = num_intervalos - i
        if i % 2 == 0:
            ax_gantt.axhspan(y_pos - 0.5, y_pos + 0.5, color='gray', alpha=0.08, zorder=0)

    y_ticks_pos = []
    y_ticks_labels = []

    for i, nombre_intervalo in enumerate(intervalos_a_validar):
        y_pos = num_intervalos - i 
        y_ticks_pos.append(y_pos)
        y_ticks_labels.append(nombre_intervalo)

        data = next((item for item in datos_a_unir if item[0] == nombre_intervalo), None)
        if not data:
            continue
        
        _, duraciones, tiempos = data
        color_actual = colores[i]

        for t_start, dur in zip(tiempos, duraciones):
            t_end = t_start + dur
            
            if t_end >= t_lim_inf and t_start <= t_lim_sup:
                # Barra principal
                ax_gantt.hlines(y=y_pos, xmin=t_start, xmax=t_end, color=color_actual, linewidth=6, alpha=0.85, zorder=3)
                # Topes delimitadores negros
                ax_gantt.plot([t_start, t_end], [y_pos, y_pos], '|', color='black', markersize=10, markeredgewidth=1.5, zorder=4)

    ax_gantt.set_yticks(y_ticks_pos)
    ax_gantt.set_yticklabels(y_ticks_labels, fontsize=10, fontweight='bold')
    ax_gantt.set_ylim(0.5, num_intervalos + 0.5)
    
    ax_gantt.set_xlabel("Tiempo Absoluto (ms)", fontsize=12, fontweight='bold')
    ax_gantt.spines['top'].set_visible(False)
    ax_gantt.spines['right'].set_visible(False)
    ax_gantt.spines['left'].set_visible(False)

    ax_lp.set_xlim(t_lim_inf, t_lim_sup)

    
    plt.tight_layout()
    plt.show(block=True)

In [ ]:
print("Iniciando validación visual multicapa...")

graficar_gantt_multicapa(
    datos_a_unir, 
    intervalos_a_validar=intervalos_a_validar, 
    senal_LP=señal_LP, 
    senal_PD=señal_PD, 
    dt=dt, 
    num_ciclos=10,       # 10 ciclos seguidos
    indice_inicio=1000   # Salta los primeros 1000 eventos (ajústalo según necesites)
)

print("Validación terminada.")